# Customer PO Extraction dengan Donut — Google Colab

Notebook ini menjalankan aplikasi **donut-po-extractor** (BUC 1 Anotasi → BUC 2 Training & Validasi → BUC 3 Ekstraksi PO) di Google Colab dengan GPU gratis.

**Sebelum mulai:** menu **Runtime → Change runtime type → T4 GPU**, lalu jalankan sel satu per satu dari atas (Shift+Enter).

Urutan:
1. Cek GPU
2. Hubungkan Google Drive (agar sampel, record, dan model tidak hilang saat runtime Colab di-reset)
3. Ambil kode dari GitHub
4. Install library
5. Jalankan aplikasi dan buka UI-nya
6. (Opsional) Training lewat CLI, ekstraksi lewat sel, test

## 1. Cek GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'Tidak ada GPU: aktifkan lewat Runtime > Change runtime type > T4 GPU'

## 2. Hubungkan Google Drive (disarankan)

Semua data aplikasi (sampel, ground truth, model, hasil ekstraksi) disimpan di `MyDrive/donut-po-data`.
Set `USE_DRIVE = False` kalau hanya ingin mencoba (data hilang saat runtime berhenti).

Catatan: setiap training run menyimpan model ±800 MB. Hapus run lama dari folder `models/runs` di Drive bila ruang penuh.

In [ ]:
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/donut-po-data'
else:
    DATA_DIR = '/content/donut-po-data'
os.makedirs(DATA_DIR, exist_ok=True)
print('Data disimpan di', DATA_DIR)

## 3. Ambil kode dari GitHub

Kalau repository **private**, buat GitHub token (Settings → Developer settings → Personal access tokens, akses *read* ke repo),
lalu simpan di Colab: ikon kunci 🔑 (Secrets) di panel kiri → nama `GITHUB_TOKEN` → aktifkan *Notebook access*.
Repository public tidak butuh token.

In [ ]:
REPO = 'awaludis/Warung'
BRANCH = 'claude/sharp-wright-7pybmh'

import os, subprocess
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass  # secret belum dibuat: repo public tidak membutuhkannya

url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if os.path.isdir('/content/Warung/.git'):
    subprocess.run(['git', '-C', '/content/Warung', 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, '/content/Warung'], check=True)

APP_DIR = '/content/Warung/donut-po-extractor'
%cd {APP_DIR}
!ls

## 4. Install library

PyTorch dengan CUDA sudah tersedia di Colab; sel ini memasang sisanya (transformers, PyMuPDF, FastAPI, ...). ±1–2 menit.

Jika muncul pesan merah **`ERROR: pip's dependency resolver ... diffusers / gradio requires huggingface-hub ...`**, abaikan saja:
itu hanya peringatan untuk paket bawaan Colab (diffusers, gradio) yang tidak dipakai aplikasi ini.
Instalasi berhasil bila baris terakhir menampilkan `CUDA: True`.

In [ ]:
!pip install -q -r requirements.txt -r requirements-dev.txt
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| CUDA:', torch.cuda.is_available())

## 5. Jalankan aplikasi dan buka UI

Pengaturan di bawah sudah disesuaikan untuk GPU T4 (16 GB). Jika muncul *CUDA out of memory* saat training,
turunkan `IMAGE_HEIGHT`/`IMAGE_WIDTH` (mis. 960×720), lalu jalankan ulang sel ini.

In [ ]:
IMAGE_HEIGHT = 1280
IMAGE_WIDTH = 960
PORT = 8000

import os, subprocess, time, urllib.request

env = dict(os.environ,
           DONUT_PO_DATA_DIR=DATA_DIR,
           DONUT_PO_IMAGE_HEIGHT=str(IMAGE_HEIGHT),
           DONUT_PO_IMAGE_WIDTH=str(IMAGE_WIDTH))
os.environ.update(env)  # juga dipakai sel CLI di bawah

# hentikan server lama bila sel ini dijalankan ulang
if 'server' in globals() and server.poll() is None:
    server.terminate(); server.wait()

log = open('/content/server.log', 'w')
server = subprocess.Popen(['python', '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', str(PORT)],
                          cwd=APP_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/api/status', timeout=2); break
    except Exception:
        time.sleep(1)
else:
    print(open('/content/server.log').read()); raise RuntimeError('Server gagal start, lihat log di atas')

from google.colab.output import eval_js
app_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
print('Aplikasi berjalan. Buka link ini di tab baru:')
print(app_url)

Link di atas hanya bisa dibuka oleh akun Google Anda dan hanya selama notebook ini aktif.

Alternatif — tampilkan aplikasi langsung di bawah sel ini:

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(PORT, height=900)

### (Opsional) Link publik untuk dibagikan ke tim

Membuat link `https://....trycloudflare.com` yang bisa dibuka siapa saja (mis. Data Annotator lain).
⚠️ Tidak ada login — siapa pun yang punya link bisa melihat dan mengubah data. Bagikan hanya ke orang yang dipercaya, dan hentikan sel ini setelah selesai.

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared
import subprocess, re, time
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('Link publik:', m.group(0)); break

## Dokumen latihan

Repo ini berisi 57 PO fiktif di `practice_docs/` (3 layout pelanggan untuk training + 1 layout tak dikenal untuk uji ekstraksi), lengkap dengan kunci jawaban. Penjelasan: `practice_docs/README.md`.

**Pilihan 1 — unduh ke komputer** untuk diunggah manual lewat UI (latihan anotasi BUC 1 dan uji ekstraksi BUC 3):

In [ ]:
!cd {APP_DIR} && rm -f /content/practice_docs.zip && zip -qr /content/practice_docs.zip practice_docs
from google.colab import files
files.download('/content/practice_docs.zip')

**Pilihan 2 — impor langsung** semua dokumen training + kunci jawaban sebagai ground truth entry yang sudah selesai (melewati anotasi manual). Jalankan **sekali saja**, lalu di Tab 1 klik **File Ground Truth Record** (muat ulang halaman UI dulu).

In [ ]:
!cd {APP_DIR} && python -m app.cli --user colab import-samples practice_docs/training

## Cara pakai

1. **Tab 1 · Anotasi Sampel** — isi nama & peran di kanan atas, unggah contoh PO + nama *customer layout*, isi field, *Simpan & tandai selesai*. Setelah tiap layout punya ≥ 2 halaman selesai: **File Ground Truth Record**.
2. **Tab 2 · Training & Validasi** — pilih record → *Bagi data & mulai training*. Training pertama mengunduh `naver-clova-ix/donut-base` (±800 MB). Setelah validasi, periksa Validation Sheet → **Rilis model**.
3. **Tab 3 · Ekstraksi PO** — unggah PO → *Ekstrak* → periksa/koreksi JSON → *Simpan* → unduh JSON.

Untuk hasil yang andal siapkan ±20–50 halaman per layout pelanggan. Di T4, training 30 epoch untuk 50 halaman memakan sekitar 15–30 menit.

⚠️ Colab memutus runtime bila tab ditutup atau idle terlalu lama. Training yang sedang berjalan akan berhenti (status *failed*), tetapi data di Google Drive tetap aman — jalankan ulang sel 3–5 lalu mulai training lagi.

## 6. (Opsional) Lewat sel, tanpa UI

Melihat ground truth record yang sudah di-file:

In [ ]:
!python -m app.cli records

Training + validasi + rilis sekaligus (log tampil langsung di sel). Ganti `RECORD_ID` dengan id dari sel di atas.
Jangan jalankan bersamaan dengan training dari UI.

In [ ]:
RECORD_ID = 'GTR-...'
EPOCHS = 30
!python -m app.cli --user colab train --record {RECORD_ID} --epochs {EPOCHS} --release

Ekstraksi satu file PO (unggah file lewat dialog), hasil disimpan sebagai Structured PO Data Record:

In [ ]:
from google.colab import files
uploaded = files.upload()
LAYOUT = ''  # nama customer layout, mis. 'PT Sumber Makmur'
for name in uploaded:
    open(f'/content/{name}', 'wb').write(uploaded[name])
    !python -m app.cli --user colab extract "/content/{name}" --layout "{LAYOUT}" --save

Log server (untuk mencari penyebab error):

In [ ]:
!tail -n 50 /content/server.log

Menjalankan test otomatis (memakai data sementara, tidak menyentuh data di Drive):

In [ ]:
!cd {APP_DIR} && python -m pytest -q